# Fine-tune Qwen2.5-Coder-3B with Unsloth
Run this in Google Colab (T4 GPU) or Kaggle.

In [ ]:
!pip install "unsloth[colab-new]" @ git+https://github.com/unslothai/unsloth.git
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes datasets

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
from datasets import load_dataset
# Ensure train.jsonl and eval.jsonl are uploaded to your Colab workspace
train_ds = load_dataset("json", data_files="train.jsonl", split="train")
eval_ds = load_dataset("json", data_files="eval.jsonl", split="train")

def format_chat(examples):
    texts = []
    for messages in examples["messages"]:
        texts.append(tokenizer.apply_chat_template(messages, tokenize=False))
    return {"text": texts}

train_ds = train_ds.map(format_chat, batched=True)
eval_ds = eval_ds.map(format_chat, batched=True)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=0, # Let it use num_train_epochs
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        evaluation_strategy="steps",
        eval_steps=50,
        optim="adamw_8bit",
        weight_decay=0.01,
        output_dir="axen-coder-outputs",
        seed=3407,
    ),
)
trainer.train()

In [ ]:
FastLanguageModel.for_inference(model)
messages = [
    {"role": "user", "content": "Write a short python function to reverse a string."}
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(inputs, max_new_tokens=128)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
model.save_pretrained_gguf("axen-coder-gguf", tokenizer, quantization_method="q4_k_m")
print("GGUF saved! Download the .gguf file from the axen-coder-gguf directory.")